## PRE_HARVEST MODEL INFERENCE TO PREDICT 2025 YIELD

In [94]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from lightgbm import LGBMRegressor
import mlflow

In [95]:
COUNTY = "plymouth"

print("Selected county:", COUNTY)

Selected county: plymouth


In [97]:
ndvi = pd.read_csv("../inference-dataset/intermediate/ndvi_until_aug01.csv")
wx = pd.read_csv("../inference-dataset/intermediate/weather_until_aug01.csv")
storm = pd.read_csv("../inference-dataset/intermediate/storm_until_aug01.csv")
h_yeild = pd.read_csv("../inference-dataset/intermediate/yield_history.csv")

# Raw data used for prediction

In [98]:
ndvi_c = ndvi[ndvi["county"] == COUNTY]
wx_c = wx[wx["county"] == COUNTY]
storm_c = storm[storm["county"] == COUNTY]
yield_c = h_yeild[h_yeild["county"] == COUNTY]

print("\n===== NDVI DATA =====")
print("Rows:", ndvi_c.shape)
print(ndvi_c)

print("\n===== WEATHER DATA =====")
print("Rows:", wx_c.shape)
print(wx_c.tail())

print("\n===== STORM DATA =====")
print("Rows:", storm_c.shape)
print(storm_c.tail())

print("\n===== HISTORICAL YIELD =====")
print("Rows:", yield_c.shape)
print(yield_c.tail())


===== NDVI DATA =====
Rows: (8, 4)
       county        date  year      NDVI
592  plymouth  2025-04-07  2025  0.255484
593  plymouth  2025-04-23  2025  0.291939
594  plymouth  2025-05-09  2025  0.284812
595  plymouth  2025-05-25  2025  0.365425
596  plymouth  2025-06-10  2025  0.556334
597  plymouth  2025-06-26  2025  0.856243
598  plymouth  2025-07-12  2025  0.897586
599  plymouth  2025-07-28  2025  0.926376

===== WEATHER DATA =====
Rows: (123, 6)
        county        date  year  temperature      rainfall  \
9220  plymouth  2025-07-28  2025    28.070919  2.845150e-03   
9221  plymouth  2025-07-29  2025    27.258957  1.637931e-02   
9222  plymouth  2025-07-30  2025    22.928874  2.951522e-02   
9223  plymouth  2025-07-31  2025    21.441649  2.241528e-03   
9224  plymouth  2025-08-01  2025    19.851054  4.351139e-07   

      evapotranspiration  
9220           -0.006152  
9221           -0.004714  
9222           -0.003506  
9223           -0.005647  
9224           -0.005695  

===

In [99]:
augset = pd.read_csv("/Users/jincyjose/PycharmProjects/453/old_capstone/Feb22 Final Experiments/inference-dataset/features_frozen/features_aug01.csv")
augseta = augset[augset["county"] == COUNTY]
augseta


,county,year,rolling_3yr_mean,ndvi_peak,ndvi_slope,heat_days_gt32,temp_anomaly,net_moisture_stress,wind_severe_days_58_cutoff
73,plymouth,2025,191.033333,0.946131,0.007096,2,-0.977232,0.008589,2.0


In [102]:
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# PATH SETUP
# ============================================================

CURRENT_DIR = Path.cwd()

MODEL_PATH = "/Users/jincyjose/PycharmProjects/453/old_capstone/Feb22 Final Experiments/exported_models/aug01/LightGBM-limited_withstorm/model.pkl"


# ============================================================
# LOAD MODEL
# ============================================================

print("\nLoading model...")

with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

print("Model loaded:", type(model))

# ============================================================
# USE EXISTING DATAFRAME
# ============================================================

df = augset.copy()

print("\nUsing dataframe augseta")
print("Data shape:", df.shape)

# clean county column
if "county" in df.columns:
    df["county"] = df["county"].astype(str).str.strip()
    df["county"] = df["county"].astype("category")

# ============================================================
# FEATURES (MUST MATCH TRAINING)
# ============================================================

FEATURES = [
    "county",
    "rolling_3yr_mean",
    "ndvi_peak",
    "ndvi_slope",
    "temp_anomaly",
    "net_moisture_stress",
    "heat_days_gt32",
    "wind_severe_days_58_cutoff"
]

# ============================================================
# RUN INFERENCE
# ============================================================

print("\nRunning inference...")

X = df[FEATURES]

# Ensure numeric columns except county
for col in X.columns:
    if col != "county":
        X[col] = pd.to_numeric(X[col], errors="coerce")

X = X.fillna(0)

# Fix LightGBM sklearn wrapper issue
if hasattr(model, "booster_"):
    booster = model.booster_
    predictions = booster.predict(X)
else:
    predictions = model.predict(X)

df["predicted_yield"] = predictions

county_df = df[df["county"].astype(str).str.lower() == COUNTY]

display(county_df[["county", "predicted_yield"]])
# ============================================================
# SAVE PREDICTIONS
# ============================================================

output_csv = OUTPUT_DIR / "predictions_aug01.csv"
df.to_csv(output_csv, index=False)
print(f"Saved predictions → {output_csv}")

print("\nUnseen inference complete.")


Loading model...
Model loaded: <class 'lightgbm.sklearn.LGBMRegressor'>

Using dataframe augseta
Data shape: (98, 9)

Running inference...


,county,predicted_yield
73,plymouth,206.190707


Saved predictions → /Users/jincyjose/PycharmProjects/453/old_capstone/Feb22 Final Experiments/inference-demo/predictions_aug01.csv

Unseen inference complete.
